# Phase 5: MongoDB Development

## Step 1: Verify MongoDB Connection

In [25]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27017/"
)

client.list_database_names()

['admin', 'churn_analytics', 'config', 'local', 'mydb']

## Step 2: Create Database

In [26]:
db = client["churn_analytics"]
db

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'churn_analytics')

## Step 3: Load Cleaned Datasets

In [27]:
import pandas as pd

telco = pd.read_csv(
    "../data/processed/clean_telco.csv"
)

retail_churn = pd.read_csv(
    "../data/processed/clean_retail_churn.csv"
)

retail_sales = pd.read_csv(
    "../data/processed/clean_retail_sales.csv"
)

## Step 4: Convert DataFrames to Documents

In [28]:
telco_records = (
    telco.to_dict(
        orient="records"
    )
)

retail_churn_records = (
    retail_churn.to_dict(
        orient="records"
    )
)

retail_sales_records = (
    retail_sales.to_dict(
        orient="records"
    )
)

## Step 5: Create Collections

In [29]:
telco_col = db["telco_customers"]

retail_churn_col = db[
    "retail_churn_customers"
]

retail_sales_col = db[
    "retail_sales_customers"
]

## Step 6: Remove Existing Data

In [30]:
telco_col.delete_many({})

retail_churn_col.delete_many({})

retail_sales_col.delete_many({})

DeleteResult({'n': 1000000, 'ok': 1.0}, acknowledged=True)

## Step 7: Insert Documents

In [31]:
_ = telco_col.insert_many(
    telco_records
)

_ = retail_churn_col.insert_many(
    retail_churn_records
)

_ = retail_sales_col.insert_many(
    retail_sales_records
)

## Step 8: Validate Row Counts

In [8]:
print(
    "Telco:",
    telco_col.count_documents({})
)

print(
    "Retail Churn:",
    retail_churn_col.count_documents({})
)

print(
    "Retail Sales:",
    retail_sales_col.count_documents({})
)

Telco: 7043
Retail Churn: 5630
Retail Sales: 1000000


## Step 9: Create MongoDB Analytics Queries

### Query 1: Telco Churn Distribution

In [9]:
list(
    telco_col.aggregate([
        {
            "$group":
            {
                "_id":"$Churn Label",
                "Customers":
                {
                    "$sum":1
                }
            }
        }
    ])
)

[{'_id': 'Yes', 'Customers': 1869}, {'_id': 'No', 'Customers': 5174}]

### Query 2: Telco Contract Type Analysis

In [10]:
list(
    telco_col.aggregate([
        {
            "$group":
            {
                "_id":"$Contract",
                "Customers":
                {
                    "$sum":1
                }
            }
        },
        {
            "$sort":
            {
                "Customers":-1
            }
        }
    ])
)

[{'_id': 'Month-to-month', 'Customers': 3875},
 {'_id': 'Two year', 'Customers': 1695},
 {'_id': 'One year', 'Customers': 1473}]

### Query 3: Average Monthly Charges by Churn

In [11]:
list(
    telco_col.aggregate([
        {
            "$group":
            {
                "_id":"$Churn Label",
                "AvgMonthlyCharges":
                {
                    "$avg":"$Monthly Charges"
                }
            }
        }
    ])
)

[{'_id': 'Yes', 'AvgMonthlyCharges': 74.44133226324237},
 {'_id': 'No', 'AvgMonthlyCharges': 61.26512369540008}]

### Retail Churn Queries

#### Query 4: Complaints vs Churn

In [12]:
list(
    retail_churn_col.aggregate([
        {
            "$group":
            {
                "_id":
                {
                    "Churn":"$Churn",
                    "Complain":"$Complain"
                },
                "Customers":
                {
                    "$sum":1
                }
            }
        }
    ])
)

[{'_id': {'Churn': 1, 'Complain': 1}, 'Customers': 508},
 {'_id': {'Churn': 0, 'Complain': 0}, 'Customers': 3586},
 {'_id': {'Churn': 0, 'Complain': 1}, 'Customers': 1096},
 {'_id': {'Churn': 1, 'Complain': 0}, 'Customers': 440}]

#### Query 5: Average Cashback

In [13]:
list(
    retail_churn_col.aggregate([
        {
            "$group":
            {
                "_id":"$Churn",
                "AvgCashback":
                {
                    "$avg":"$CashbackAmount"
                }
            }
        }
    ])
)

[{'_id': 0, 'AvgCashback': 180.63520290474156},
 {'_id': 1, 'AvgCashback': 160.3709282700422}]

#### Query 6: Average Satisfaction

In [14]:
list(
    retail_churn_col.aggregate([
        {
            "$group":
            {
                "_id":"$Churn",
                "AvgSatisfaction":
                {
                    "$avg":"$SatisfactionScore"
                }
            }
        }
    ])
)

[{'_id': 1, 'AvgSatisfaction': 3.390295358649789},
 {'_id': 0, 'AvgSatisfaction': 3.001281503630927}]

### Retail Sales Queries

#### Query 7: Loyalty Program Analysis

In [15]:
list(
    retail_sales_col.aggregate([
        {
            "$group":
            {
                "_id":
                {
                    "Loyalty":
                    "$loyalty_program",
                    "Churned":
                    "$churned"
                },
                "Customers":
                {
                    "$sum":1
                }
            }
        }
    ])
)

[{'_id': {'Loyalty': 'Yes', 'Churned': 'No'}, 'Customers': 250063},
 {'_id': {'Loyalty': 'No', 'Churned': 'Yes'}, 'Customers': 250080},
 {'_id': {'Loyalty': 'No', 'Churned': 'No'}, 'Customers': 250208},
 {'_id': {'Loyalty': 'Yes', 'Churned': 'Yes'}, 'Customers': 249649}]

#### Query 8: Revenue by Product Category

In [16]:
list(
    retail_sales_col.aggregate([
        {
            "$group":
            {
                "_id":
                "$product_category",
                "Revenue":
                {
                    "$sum":
                    "$total_sales"
                }
            }
        },
        {
            "$sort":
            {
                "Revenue":-1
            }
        },
        {
            "$limit":10
        }
    ])
)

[{'_id': 'Toys', 'Revenue': 1013781081.63},
 {'_id': 'Clothing', 'Revenue': 1011085885.32},
 {'_id': 'Groceries', 'Revenue': 1011072707.78},
 {'_id': 'Electronics', 'Revenue': 1010576351.51},
 {'_id': 'Furniture', 'Revenue': 1009543738.9}]

#### Query 9: Customer Support Calls

In [17]:
list(
    retail_sales_col.aggregate([
        {
            "$group":
            {
                "_id":"$churned",
                "AvgSupportCalls":
                {
                    "$avg":
                    "$customer_support_calls"
                }
            }
        }
    ])
)

[{'_id': 'No', 'AvgSupportCalls': 9.49257502433681},
 {'_id': 'Yes', 'AvgSupportCalls': 9.4999669821043}]

## Step 10: Compare SQL vs MongoDB

### Create SQL Results Dictionary

In [22]:
import sqlite3

conn = sqlite3.connect(
    "../sqlite/churn_analysis.db"
)

sql_results = {}

sql_results["Telco Churn Distribution"] = pd.read_sql(
    "SELECT [Churn Label], COUNT(*) AS Customers FROM telco_customers GROUP BY [Churn Label]",
    conn
)

sql_results["Retail Churn Distribution"] = pd.read_sql(
    "SELECT Churn, COUNT(*) AS Customers FROM retail_churn_customers GROUP BY Churn",
    conn
)

sql_results["Retail Sales Churn Distribution"] = pd.read_sql(
    "SELECT churned, COUNT(*) AS Customers FROM retail_sales_customers GROUP BY churned",
    conn
)

### Create Equivalent MongoDB Results

In [32]:
mongo_results = {}

mongo_results["Telco Churn Distribution"] = pd.DataFrame(
    list(
        telco_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn Label",
                    "Customers": {"$sum": 1}
                }
            }
        ])
    )
)

mongo_results["Retail Churn Distribution"] = pd.DataFrame(
    list(
        retail_churn_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn",
                    "Customers": {"$sum": 1}
                }
            }
        ])
    )
)

mongo_results["Retail Sales Churn Distribution"] = pd.DataFrame(
    list(
        retail_sales_col.aggregate([
            {
                "$group": {
                    "_id": "$churned",
                    "Customers": {"$sum": 1}
                }
            }
        ])
    )
)

### Generate Comparison Report

In [33]:
from pathlib import Path

comparison_file = Path(
    "../reports/sql_vs_mongodb_comparison.md"
)

with open(
    comparison_file,
    "w",
    encoding="utf-8"
) as f:

    f.write("# SQL vs MongoDB Comparison\n\n")

    f.write(
        "This report compares equivalent analytics performed using SQLite and MongoDB.\n\n"
    )

    for query_name in sql_results.keys():

        f.write(
            f"## {query_name}\n\n"
        )

        f.write("### SQLite Result\n\n")

        f.write(
            sql_results[query_name]
            .to_markdown(index=False)
        )

        f.write("\n\n")

        f.write("### MongoDB Result\n\n")

        f.write(
            mongo_results[query_name]
            .to_markdown(index=False)
        )

        f.write("\n\n")

        f.write(
            "### Observation\n\n"
        )

        f.write(
            "- Both systems produced equivalent business results.\n"
        )

        f.write(
            "- SQLite used SQL GROUP BY statements.\n"
        )

        f.write(
            "- MongoDB used aggregation pipelines.\n\n"
        )

        f.write("---\n\n")

print(
    f"Report created: {comparison_file}"
)

Report created: ..\reports\sql_vs_mongodb_comparison.md


### Generate Technology Comparison Table

In [34]:
with open(
    comparison_file,
    "a",
    encoding="utf-8"
) as f:

    f.write("## Technology Comparison\n\n")

    f.write("""
| Feature | SQLite | MongoDB |
|----------|----------|----------|
| Database Type | Relational | NoSQL Document |
| Schema | Fixed | Flexible |
| Query Language | SQL | Aggregation Pipeline |
| Joins | Strong Support | Limited |
| JSON Storage | Limited | Native |
| Transaction Data | Excellent | Good |
| Customer Profiles | Good | Excellent |
| Analytics | Excellent | Excellent |
| Scalability | Moderate | High |
""")

### Generate Project Findings

In [35]:
with open(
    comparison_file,
    "a",
    encoding="utf-8"
) as f:

    f.write("\n\n## Project Conclusions\n\n")

    f.write(
        "- SQLite was well suited for structured customer analytics and reporting.\n"
    )

    f.write(
        "- MongoDB provided flexible document storage for customer profiles.\n"
    )

    f.write(
        "- Both technologies produced consistent churn analysis results.\n"
    )

    f.write(
        "- SQLite was easier for tabular business reporting.\n"
    )

    f.write(
        "- MongoDB was better suited for semi-structured customer records.\n"
    )

## Step 11: Generate MongoDB Analysis Report

### Store MongoDB Query Results

In [18]:
mongo_results = {}

# ==================================================
# TELCO
# ==================================================

mongo_results["Telco Churn Distribution"] = pd.DataFrame(
    list(
        telco_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn Label",
                    "Customers": {"$sum": 1}
                }
            }
        ])
    )
)

mongo_results["Telco Contract Distribution"] = pd.DataFrame(
    list(
        telco_col.aggregate([
            {
                "$group": {
                    "_id": "$Contract",
                    "Customers": {"$sum": 1}
                }
            },
            {
                "$sort": {
                    "Customers": -1
                }
            }
        ])
    )
)

mongo_results["Telco Average Monthly Charges"] = pd.DataFrame(
    list(
        telco_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn Label",
                    "AvgMonthlyCharges": {
                        "$avg": "$Monthly Charges"
                    }
                }
            }
        ])
    )
)

# ==================================================
# RETAIL CHURN
# ==================================================

mongo_results["Retail Complaints vs Churn"] = pd.DataFrame(
    list(
        retail_churn_col.aggregate([
            {
                "$group": {
                    "_id": {
                        "Churn": "$Churn",
                        "Complain": "$Complain"
                    },
                    "Customers": {
                        "$sum": 1
                    }
                }
            }
        ])
    )
)

mongo_results["Retail Average Cashback"] = pd.DataFrame(
    list(
        retail_churn_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn",
                    "AvgCashback": {
                        "$avg": "$CashbackAmount"
                    }
                }
            }
        ])
    )
)

mongo_results["Retail Average Satisfaction"] = pd.DataFrame(
    list(
        retail_churn_col.aggregate([
            {
                "$group": {
                    "_id": "$Churn",
                    "AvgSatisfaction": {
                        "$avg": "$SatisfactionScore"
                    }
                }
            }
        ])
    )
)

# ==================================================
# RETAIL SALES
# ==================================================

mongo_results["Retail Sales Loyalty Analysis"] = pd.DataFrame(
    list(
        retail_sales_col.aggregate([
            {
                "$group": {
                    "_id": {
                        "Loyalty": "$loyalty_program",
                        "Churned": "$churned"
                    }
                }
            }
        ])
    )
)

### Generate MongoDB Report

In [40]:
from pathlib import Path

report_file = Path(
    "../reports/mongodb_analysis_summary.md"
)

with open(
    report_file,
    "w",
    encoding="utf-8"
) as f:

    f.write("# MongoDB Analysis Summary\n\n")

    f.write("## Database Overview\n\n")

    f.write(
        f"- Telco Documents: {telco_col.count_documents({}):,}\n"
    )

    f.write(
        f"- Retail Churn Documents: {retail_churn_col.count_documents({}):,}\n"
    )

    f.write(
        f"- Retail Sales Documents: {retail_sales_col.count_documents({}):,}\n\n"
    )

    f.write("---\n\n")

    for query_name, result in mongo_results.items():

        f.write(
            f"## {query_name}\n\n"
        )

        f.write(
            result.to_markdown(
                index=False
            )
        )

        f.write("\n\n")

        if len(result) > 0:

            f.write(
                "### Observation\n\n"
            )

            try:

                first_row = result.iloc[0]

                f.write(
                    f"- Top result: {first_row.to_dict()}\n\n"
                )

            except:
                pass

        f.write("## Collection Statistics\n\n")

        f.write("| Collection | Documents |\n")
        f.write("|------------|-----------|\n")

        f.write(f"| telco_customers | {telco_col.count_documents({}):,} |\n")

        f.write(f"| retail_churn_customers | {retail_churn_col.count_documents({}):,} |\n")

        f.write(f"| retail_sales_customers | {retail_sales_col.count_documents({}):,} |\n")

        f.write("\n")

        f.write("---\n\n")

print(
    f"MongoDB report created: {report_file}"
)

MongoDB report created: ..\reports\mongodb_analysis_summary.md


## Step 12: Phase 5 Validation

In [19]:
print(
    db.list_collection_names()
)

['retail_churn_customers', 'retail_sales_customers', 'telco_customers']
